Goal of 05_Build_FactSales.ipynb

We'll create the final FactSales table that contains:
Foreign Keys (to all dimensions)
Measures (SalesAmount, Quantity, Cost, Profit, etc.)
Transaction-level data

In [1]:
#Step 1 — Load Everything
import pandas as pd

sales = pd.read_csv("../data/processed/transactions_enriched.csv")
sales["InvoiceDate"] = pd.to_datetime(sales["InvoiceDate"])

#load the dimension
dim_product = pd.read_csv("../data/warehouse/DimProduct.csv")
dim_customer = pd.read_csv("../data/warehouse/DimCustomer.csv")
dim_date = pd.read_csv("../data/warehouse/DimDate.csv")
dim_country = pd.read_csv("../data/warehouse/DimCountry.csv")

C:\Users\محمد الرويلي\AppData\Local\Temp\ipykernel_20820\884517713.py:4: DtypeWarning: Columns (0: InvoiceNo) have mixed types. Specify dtype option on import or set low_memory=False.
  sales = pd.read_csv("../data/processed/transactions_enriched.csv")


In [2]:
#Step 2 — Create DateKey
sales["DateKey"] = (
    sales["InvoiceDate"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

In [3]:
#Step 3 — Handle Unknown Customers
sales["CustomerID"] = (
    sales["CustomerID"]
    .fillna(0)
    .astype(int)
)

In [4]:
#Step 4 — Merge ProductKey
sales = sales.merge(
    dim_product[
        ["ProductKey", "StockCode"]
    ],
    on="StockCode",
    how="left"
)

In [5]:
#Step 5 — Merge CustomerKey
sales = sales.merge(
    dim_customer[
        ["CustomerKey", "CustomerID"]
    ],
    on="CustomerID",
    how="left"
)

In [6]:
#Step 6 — Merge CountryKey
sales = sales.merge(
    dim_country[
        ["CountryKey", "Country"]
    ],
    on="Country",
    how="left"
)

In [7]:
#Step 7 — Validate DateKey
sales["DateKey"].isin(dim_date["DateKey"]).all()

np.True_

In [8]:
#Step 8 — Create FactKey
sales.insert(
    0,
    "SalesKey",
    range(1, len(sales) + 1)
)

In [9]:
#Step 9 — Keep Only Warehouse Columns
fact_sales = sales[
    [
        "SalesKey",
        "InvoiceNo",
        "DateKey",
        "CustomerKey",
        "ProductKey",
        "CountryKey",
        "Quantity",
        "UnitPrice",
        "DiscountPercent",
        "SalesAmount",
        "CostRatio",
        "UnitCost",
        "TotalCost",
        "Profit",
        "ProfitMargin"
    ]
]

In [10]:
#Step 10 — Validate the Warehouse

#Missing Product Keys
fact_sales["ProductKey"].isna().sum()

#Missing Customer Keys
fact_sales["CustomerKey"].isna().sum()

#Missing Country Keys
fact_sales["CountryKey"].isna().sum()

np.int64(0)

In [11]:
#Step 11 — Final Column Order
fact_sales = sales[
    [
        "SalesKey",
        "InvoiceNo",
        "DateKey",
        "CustomerKey",
        "ProductKey",
        "CountryKey",
        "Quantity",
        "UnitPrice",
        "DiscountPercent",
        "SalesAmount",
        "CostRatio",
        "UnitCost",
        "TotalCost",
        "Profit",
        "ProfitMargin"
    ]
]

In [12]:
#Step 12 — Save
fact_sales.to_csv(
    "../data/warehouse/FactSales.csv",
    index=False
)

Warehouse Validation

In [13]:
#1. Basic Information
print("=" * 50)
print("FACT TABLE VALIDATION")
print("=" * 50)

print(f"Rows: {len(fact_sales):,}")
print(f"Columns: {len(fact_sales.columns)}")

FACT TABLE VALIDATION
Rows: 578,575
Columns: 15


In [14]:
#2. Missing Values
print("\nMissing Values")

print(fact_sales.isnull().sum())


Missing Values
SalesKey           0
InvoiceNo          0
DateKey            0
CustomerKey        0
ProductKey         0
CountryKey         0
Quantity           0
UnitPrice          0
DiscountPercent    0
SalesAmount        0
CostRatio          0
UnitCost           0
TotalCost          0
Profit             0
ProfitMargin       0
dtype: int64


In [15]:
#3. Duplicate Sales Keys
duplicates = fact_sales["SalesKey"].duplicated().sum()

print(f"Duplicate SalesKeys: {duplicates}")

Duplicate SalesKeys: 0


In [16]:
#4. Validate Product Keys
invalid_products = (
    ~fact_sales["ProductKey"]
    .isin(dim_product["ProductKey"])
).sum()

print(f"Invalid ProductKeys: {invalid_products}")

Invalid ProductKeys: 0


In [17]:
#5. Validate Customer Keys
invalid_customers = (
    ~fact_sales["CustomerKey"]
    .isin(dim_customer["CustomerKey"])
).sum()

print(f"Invalid CustomerKeys: {invalid_customers}")

Invalid CustomerKeys: 0


In [18]:
#6. Validate Country Keys
invalid_countries = (
    ~fact_sales["CountryKey"]
    .isin(dim_country["CountryKey"])
).sum()

print(f"Invalid CountryKeys: {invalid_countries}")

Invalid CountryKeys: 0


In [19]:
#7. Validate Date Keys
invalid_dates = (
    ~fact_sales["DateKey"]
    .isin(dim_date["DateKey"])
).sum()

print(f"Invalid DateKeys: {invalid_dates}")

Invalid DateKeys: 0


In [20]:
#8. Business Validation
print("\nBusiness Validation")

print(f"Total Sales: {fact_sales['SalesAmount'].sum():,.2f}")

print(f"Total Profit: {fact_sales['Profit'].sum():,.2f}")

print(f"Average Profit Margin: {fact_sales['ProfitMargin'].mean():.2f}%")



Business Validation
Total Sales: 11,948,795.43
Total Profit: 4,218,996.51
Average Profit Margin: 35.07%


In [21]:
#9. Final Validation Report
print("\n" + "=" * 50)
print("WAREHOUSE VALIDATION SUMMARY")
print("=" * 50)

checks = {
    "Missing Values": fact_sales.isnull().sum().sum() == 0,
    "Duplicate SalesKeys": duplicates == 0,
    "Valid ProductKeys": invalid_products == 0,
    "Valid CustomerKeys": invalid_customers == 0,
    "Valid CountryKeys": invalid_countries == 0,
    "Valid DateKeys": invalid_dates == 0,
}

for check, result in checks.items():
    status = "PASS" if result else "FAIL"
    print(f"{check:<25} {status}")


WAREHOUSE VALIDATION SUMMARY
Missing Values            PASS
Duplicate SalesKeys       PASS
Valid ProductKeys         PASS
Valid CustomerKeys        PASS
Valid CountryKeys         PASS
Valid DateKeys            PASS


In [22]:
fact_sales[fact_sales["ProfitMargin"].isna()]

,SalesKey,InvoiceNo,DateKey,CustomerKey,ProductKey,CountryKey,Quantity,UnitPrice,DiscountPercent,SalesAmount,CostRatio,UnitCost,TotalCost,Profit,ProfitMargin


In [23]:
sales[sales["ProfitMargin"].isna()][
    [
        "InvoiceNo",
        "StockCode",
        "Quantity",
        "UnitPrice",
        "SalesAmount",
        "UnitCost",
        "TotalCost",
        "Profit",
        "ProfitMargin"
    ]
]

,InvoiceNo,StockCode,Quantity,UnitPrice,SalesAmount,UnitCost,TotalCost,Profit,ProfitMargin
